# Q-BioDiag: Quantum AI for Biomedical Diagnostics

Updated hackathon notebook with claim guardrails, 5-fold CV, and classifier-level noise resilience.


## Claim guardrail
Say: **100% accuracy on the 40-sample held-out evaluation set in this prototype pipeline.** Do not claim clinical accuracy or IBM Quantum hardware accuracy unless a real QPU job is completed.


In [ ]:
from qbio_diagnostics.data import generate_synthetic_biomedical_patches, save_montage
dataset = generate_synthetic_biomedical_patches(n_samples=160)
save_montage(dataset, "../outputs/sample_patches.png")
dataset.images.shape, dataset.labels.mean()


## Compress images into 4 qubit features


In [ ]:
from qbio_diagnostics.features import make_qubit_features
bundle = make_qubit_features(dataset.images, dataset.labels, qubits=4)
bundle.x_train.shape, bundle.x_test.shape, bundle.feature_names


## Classical baseline vs quantum-kernel model
The preferred path uses Qiskit `ZZFeatureMap + FidelityQuantumKernel + QSVC`. If Qiskit is unavailable, the fallback is still a **4-qubit reference quantum-kernel simulator**, not a classical polynomial SVM.


In [ ]:
from qbio_diagnostics.train_classical import train_classical_svm
from qbio_diagnostics.train_qsvc import train_quantum_qsvc
classical_model, c_pred, c_score, c_metrics = train_classical_svm(bundle.x_train, bundle.y_train, bundle.x_test, bundle.y_test)
q_model, q_pred, q_score, q_metrics, backend = train_quantum_qsvc(bundle.x_train, bundle.y_train, bundle.x_test, bundle.y_test)
c_metrics, q_metrics, backend


## Save confusion matrix and ROC


In [ ]:
from qbio_diagnostics.metrics import save_confusion_matrix, save_roc_curve
save_confusion_matrix(bundle.y_test, q_pred, "../outputs/quantum_confusion_matrix.png", "4-Qubit Quantum-Kernel Confusion Matrix")
save_roc_curve(bundle.y_test, q_score, "../outputs/quantum_roc_curve.png", "4-Qubit Quantum-Kernel ROC Curve")


## 5-fold stratified cross-validation


In [ ]:
from qbio_diagnostics.validation import run_stratified_5fold_cv
cv = run_stratified_5fold_cv(dataset.images, dataset.labels, qubits=4)
cv["reference_quantum_kernel_svm"]["mean_accuracy"], cv["reference_quantum_kernel_svm"]["std_accuracy"]


## Noise-resilience benchmark
This compares ideal, noisy, and ZNE-estimated quantum-kernel classification using a reference density-matrix simulator. IBM hardware status remains **not_run** unless you run a real QPU job.


In [ ]:
from qbio_diagnostics.noise_resilience import run_noise_classification_benchmark
noise = run_noise_classification_benchmark(bundle.x_train, bundle.y_train, bundle.x_test, bundle.y_test)
noise


## Pinecone/local case retrieval memory


In [ ]:
from qbio_diagnostics.pinecone_store import upsert_cases
upsert_cases(bundle.x_train, bundle.y_train, out_dir="../outputs", use_pinecone=False)
